# **Logical Reasoning for Planning**
**Using an LLM to Construct and Test a Simple Planning Agent**


## **Task 0: Understand the Planning Problem**

**(a) Initial state**

I = {At(Robot, A), At(Package, A)}

**(b) Goal**

G = {At(Package, C)}

**(c) Actions**

Move (between connected locations: A↔B, B↔C), PickUp(Package, loc), Drop(Package, loc)

**(d) Preconditions and effects**

| Action | Preconditions | Effects |
|---|---|---|
| Move(X, Y) | At(Robot, X) | ¬At(Robot, X), At(Robot, Y) |
| PickUp(Package, L) | At(Robot, L), At(Package, L) | ¬At(Package, L), Holding(Package) |
| Drop(Package, L) | At(Robot, L), Holding(Package) | ¬Holding(Package), At(Package, L) |

**Applicability in I**
- **PickUp(Package, A): applicable.** Both preconditions, At(Robot, A) and At(Package, A), are in I.
- **Drop(Package, C): not applicable.** It needs At(Robot, C), which is false since the robot is at A. It also needs Holding(Package), which is false.

An action is applicable only if S ⊨ Preconditions(a), not merely because it appears in the action list.

## **Task 1: Construct a Plan by Hand**

Plan: PickUp(Package, A) → Move(A, B) → Move(B, C) → Drop(Package, C)

| State | Facts |
|---|---|
| S0 | At(Robot, A), At(Package, A) |
| S1 (after PickUp(Package, A)) | At(Robot, A), Holding(Package) |
| S2 (after Move(A, B)) | At(Robot, B), Holding(Package) |
| S3 (after Move(B, C)) | At(Robot, C), Holding(Package) |
| S4 (after Drop(Package, C)) | At(Robot, C), At(Package, C) |

## **Task 2: Ask an LLM to Implement the Planner**

**Prompt:**

I want to implement a simple planning agent in Python.
Represent a state as a set of logical propositions.
Each action should contain:
- a name;
- positive preconditions;
- negative preconditions;
- positive effects;
- negative effects.

An action is applicable if all of its preconditions are satisfied by the current state.

When an action is applied:
1. remove its negative effects from the state;
2. add its positive effects to the state.

Use breadth-first search to find a sequence of actions that achieves a specified goal.

The program should also:
- detect when no plan exists;
- print the resulting sequence of actions;
- print the states reached after each action.

Explain the implementation and identify any assumptions you make.

In [1]:
from collections import deque
from dataclasses import dataclass

@dataclass(frozen=True)
class Action:
    name: str
    pos_pre: frozenset = frozenset()   # must be IN the state
    neg_pre: frozenset = frozenset()   # must NOT be in the state
    pos_eff: frozenset = frozenset()   # added
    neg_eff: frozenset = frozenset()   # removed

def make_action(name, pos_pre=(), neg_pre=(), pos_eff=(), neg_eff=()):
    return Action(name, frozenset(pos_pre), frozenset(neg_pre),
                  frozenset(pos_eff), frozenset(neg_eff))

def applicable(state, action):
    """Logical check: S |= Preconditions(a)"""
    return action.pos_pre <= state and action.neg_pre.isdisjoint(state)

def apply_action(state, action):
    """S' = (S - neg_eff) U pos_eff"""
    return (state - action.neg_eff) | action.pos_eff

def bfs_plan(initial, goal, actions):
    initial, goal = frozenset(initial), frozenset(goal)
    if goal <= initial:
        return [], [initial]
    frontier = deque([(initial, [], [initial])])
    visited = {initial}
    while frontier:
        state, plan, states = frontier.popleft()
        for a in actions:
            if applicable(state, a):                 # logic
                new_state = apply_action(state, a)
                if new_state in visited:
                    continue
                visited.add(new_state)
                new_plan = plan + [a.name]
                new_states = states + [new_state]
                if goal <= new_state:                # goal test
                    return new_plan, new_states
                frontier.append((new_state, new_plan, new_states))  # search
    return None, None

def show_result(plan, states):
    if plan is None:
        print("No plan found")
        return
    print("Plan:", " -> ".join(plan) if plan else "(empty plan)")
    print("S0:", sorted(states[0]))
    for i, (a, s) in enumerate(zip(plan, states[1:]), 1):
        print(f"S{i} after {a}: {sorted(s)}")

In [3]:
def warehouse_actions(include_pickup=True, extra_moves=()):
    acts = []
    # Move between connected locations
    edges = [("A","B"), ("B","A"), ("B","C"), ("C","B")] + list(extra_moves)
    for x, y in edges:
        acts.append(make_action(f"Move({x},{y})",
                                pos_pre={f"At(Robot,{x})"},
                                pos_eff={f"At(Robot,{y})"},
                                neg_eff={f"At(Robot,{x})"}))
    for loc in ["A","B","C"]:
        if include_pickup:
            acts.append(make_action(f"PickUp(Package,{loc})",
                                    pos_pre={f"At(Robot,{loc})", f"At(Package,{loc})"},
                                    pos_eff={"Holding(Package)"},
                                    neg_eff={f"At(Package,{loc})"}))
        acts.append(make_action(f"Drop(Package,{loc})",
                                pos_pre={f"At(Robot,{loc})", "Holding(Package)"},
                                pos_eff={f"At(Package,{loc})"},
                                neg_eff={"Holding(Package)"}))
    return acts

I = {"At(Robot,A)", "At(Package,A)"}
G = {"At(Package,C)"}

**Think About It: where the specification appears in the code**

- **Preconditions: When is an action applicable?**

In `applicable()`: `action.pos_pre <= state` and `action.neg_pre.isdisjoint(state)`.

- **Effects: How does the state change?**

In `apply_action()`: `(state - neg_eff) | pos_eff`.

- **Goal: When does planning terminate?**

At the check `goal <= new_state` inside `bfs_plan()`.

- **BFS: How are alternatives explored?**

`deque` used as a FIFO queue (`popleft`/`append`), plus a `visited` set so each state is expanded once. Shorter plans are found first.

Assumptions: closed-world (anything not in the state is false), actions are deterministic, and the goal contains only positive facts.

## **Task 3: Test the Generated Planner**

In [4]:
plan, states = bfs_plan(I, G, warehouse_actions())
show_result(plan, states)

Plan: PickUp(Package,A) -> Move(A,B) -> Move(B,C) -> Drop(Package,C)
S0: ['At(Package,A)', 'At(Robot,A)']
S1 after PickUp(Package,A): ['At(Robot,A)', 'Holding(Package)']
S2 after Move(A,B): ['At(Robot,B)', 'Holding(Package)']
S3 after Move(B,C): ['At(Robot,C)', 'Holding(Package)']
S4 after Drop(Package,C): ['At(Package,C)', 'At(Robot,C)']


In [5]:
def validate_plan(initial, goal, plan, actions):
    by_name = {a.name: a for a in actions}
    state = frozenset(initial)
    for name in plan:
        a = by_name.get(name)
        if a is None:
            print(f"INVALID: unknown action {name}"); return False
        if not applicable(state, a):
            print(f"INVALID: {name} preconditions not met in {sorted(state)}"); return False
        state = apply_action(state, a)
    ok = frozenset(goal) <= state
    print("VALID" if ok else f"INVALID: goal not reached, final state {sorted(state)}")
    return ok

validate_plan(I, G, plan, warehouse_actions())

VALID


True

In [6]:
print("Test B: no PickUp action")
planB, statesB = bfs_plan(I, G, warehouse_actions(include_pickup=False))
show_result(planB, statesB)

Test B: no PickUp action
No plan found


In [7]:
print("Test C1: extra shortcut moves A<->C (robot moves, package does not)")
actsC = warehouse_actions(extra_moves=[("A","C"), ("C","A")])
planC, statesC = bfs_plan(I, G, actsC)
show_result(planC, statesC)
validate_plan(I, G, planC, actsC)

print("\nTest C2: goal is At(Robot,C) only (robot position, not package)")
planC2, statesC2 = bfs_plan(I, {"At(Robot,C)"}, actsC)
show_result(planC2, statesC2)
print("Package at C in final state?", "At(Package,C)" in statesC2[-1])

Test C1: extra shortcut moves A<->C (robot moves, package does not)
Plan: PickUp(Package,A) -> Move(A,C) -> Drop(Package,C)
S0: ['At(Package,A)', 'At(Robot,A)']
S1 after PickUp(Package,A): ['At(Robot,A)', 'Holding(Package)']
S2 after Move(A,C): ['At(Robot,C)', 'Holding(Package)']
S3 after Drop(Package,C): ['At(Package,C)', 'At(Robot,C)']
VALID

Test C2: goal is At(Robot,C) only (robot position, not package)
Plan: Move(A,C)
S0: ['At(Package,A)', 'At(Robot,A)']
S1 after Move(A,C): ['At(Package,A)', 'At(Robot,C)']
Package at C in final state? False


**Results:**

| Test | Initial state | Goal | Plan found? | Plan | Valid? |
|---|---|---|---|---|---|
| A: solvable | {At(Robot,A), At(Package,A)} | {At(Package,C)} | Yes | PickUp(Package,A), Move(A,B), Move(B,C), Drop(Package,C) | Yes (checked by validate_plan and by hand) |
| B: no PickUp | same | {At(Package,C)} | No: "No plan found" | – | n/a, correctly reports failure |
| C1: shortcut moves | same | {At(Package,C)} | Yes | PickUp(Package,A), Move(A,C), Drop(Package,C) | Yes, still needs PickUp and Drop |
| C2: robot-only goal | same | {At(Robot,C)} | Yes | Move(A,C) | Yes for this goal, but package is NOT at C |

Conclusion: the planner does not confuse the robot reaching C with the package reaching C, because the goal is tested against the specific fact At(Package,C).

## **Task 4: Logic and Search**

Completed flow:

Current state → Check action preconditions → **Is S ⊨ Preconditions(a)? If yes, action is applicable** → Generate successor state S' = Apply(S, a) → Search over alternatives → Goal?

Explanation: Logical reasoning is used to decide whether an action is applicable (S ⊨ Preconditions(a)) and to compute how it changes the world (remove negative effects, add positive effects). Search decides which of the applicable actions to try, and in what order. BFS keeps a queue of states, expands them level by level, and stops when a state satisfies the goal. Logic determines what is possible; search determines what to try. Without logic, search would try impossible actions; without search, logic would only tell us what is possible, not which sequence reaches the goal.

## **Task 5: Can the LLM Verify Its Own Plan?**

**Prompt:**

For every action in the plan, identify its preconditions and show that those preconditions are satisfied in the state in which the action is executed.

In [ ]:
# Replay the plan and print what is true before each action
actions = warehouse_actions()
by_name = {a.name: a for a in actions}
state = frozenset(I)
for name in plan:
    a = by_name[name]
    print(f"{name}")
    print("  needs:", sorted(a.pos_pre), "| holds in current state:", applicable(state, a))
    state = apply_action(state, a)
    print("  new state:", sorted(state))

Trust (b), the independently executed state transitions, more. The program computes each state by applying the exact set operations, so it cannot 'assume' a precondition is true. The LLM's explanation is generated text. It can sound convincing and still contain a wrong state or a skipped precondition. A generated explanation is not the same as an independent verification.

## **Reflection Questions**

1. **Why specify preconditions and effects first?**

It gives the LLM a precise, unambiguous spec, so the code matches the intended semantics instead of the LLM guessing them. It also gives me something to check the output against.

2. **Error without precondition checks:**

The planner could apply Drop(Package, C) while the robot is at A, or PickUp when the package is elsewhere, producing "plans" that teleport the package to C.

3. **Why "looks reasonable" is not valid:**

A plan is valid only if every action's preconditions hold in the state where it is executed and the final state satisfies the goal. A plausible-looking sequence can violate one step (e.g. the handout's PickUp(Package, B) example).

4. **What the LLM contributed:**

The Python structure (Action class, applicability test, state update, BFS loop, printing).

5. **What I verified independently:**

The plan by hand (Task 1), the state transitions with validate_plan, the three tests, and the correctness of the applicability and effect logic.

6. **Where logic is used:**

Checking S ⊨ Preconditions(a), computing the successor state, and testing S ⊨ G.

7. **Planning and search:**

Planning is search over the state space: states are nodes, applicable actions are edges, and the goal test is the termination condition. BFS here is the same BFS from the previous module, with logic generating the neighbours.

## **Task 6: Prolog as a Plan Verifier**

In [8]:
!apt-get install -y swi-prolog > /dev/null 2>&1
!swipl --version

SWI-Prolog version 9.0.4 for x86_64-linux


In [11]:
%%writefile planner.pl
connected(a,b).
connected(b,a).
connected(b,c).
connected(c,b).

can_move(X,Y) :- connected(X,Y).
valid_move(X,Y) :- connected(X,Y).

Overwriting planner.pl


In [12]:
import subprocess
def query(goal, file="planner.pl"):
    r = subprocess.run(["swipl", "-q", "-g",
                        f"({goal} -> writeln(true) ; writeln(false))",
                        "-t", "halt", file],
                       capture_output=True, text=True)
    print(f"?- {goal}.   =>", r.stdout.strip() or r.stderr.strip())

for q in ["can_move(a,b)", "can_move(a,c)",
          "valid_move(a,b)", "valid_move(b,c)", "valid_move(a,c)"]:
    query(q)

?- can_move(a,b).   => true
?- can_move(a,c).   => false
?- valid_move(a,b).   => true
?- valid_move(b,c).   => true
?- valid_move(a,c).   => false


(a) `can_move(a,b)` is true because the fact `connected(a,b)` exists and the rule `can_move(X,Y) :- connected(X,Y)` lets Prolog derive it.

(b) `can_move(a,c)` is not established because there is no fact `connected(a,c)` and no rule that derives it (closed-world assumption: what can't be proven is treated as false).

(c) The rule is the logical implication Connected(X,Y) → CanMove(X,Y), read as: if X is connected to Y, then the robot can move from X to Y.

## **Task 7: Using Prolog to Check a Proposed Plan**

`valid_move(a,b)` and `valid_move(b,c)` succeed; `valid_move(a,c)` fails.

Challenge: if the Python planner proposed Move(a,c), Prolog rejects it, since the warehouse knowledge has no connection a→c. The Python program generated the candidate action, and Prolog independently checked it against the logical description: Generate → Independent verification.

## **Task 8: Connect Prolog to Logical Reasoning**

In [16]:
%%writefile road.pl
wet_road.
slippery :- wet_road.
reduce_speed :- slippery.

Overwriting road.pl


In [15]:
query("reduce_speed", file="road.pl")

?- reduce_speed.   => true


`reduce_speed` succeeds because Prolog proves it by chaining: `reduce_speed` needs `slippery`, which needs `wet_road`, which is a fact.

wet_road ⇒ (wet_road → slippery) ⇒ (slippery → reduce_speed) ⇒ reduce_speed

Prolog Reflection:

1. A **fact** is unconditionally true (`wet_road.`). A **rule** gives a conclusion that holds if its conditions hold (`slippery :- wet_road.`).
2. A query asks whether the statement follows from the knowledge base. Prolog answers true if it can derive it from the facts and rules.
3. It is a separate system that checks the plan against an explicit logical description, so errors in the Python code don't carry over into the check.
4. An independent verifier doesn't share the generator's mistakes or its tendency to sound plausible, so it can catch invalid output that the LLM would not flag itself.